In [1]:
import gc

import lightgbm as lgb
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_log_error

In [6]:
from sklearn.linear_model import LinearRegression
from pathlib import Path
linear = LinearRegression()

DATA_DIR = Path("../data_classic")

train_files = [
    DATA_DIR / "train_-1_predict_from_2025-08-17.parquet",
    DATA_DIR / "train_0_predict_from_2025-09-16.parquet",
    DATA_DIR / "train_1_predict_from_2025-10-16.parquet",
    DATA_DIR / "train_2_predict_from_2025-11-15.parquet",
]

validation_file = (
    DATA_DIR
    / "train_3_predict_from_2025-12-15.parquet"
)

independent_test_file = (
    DATA_DIR
    / "test_for_us_predict_from_2026-01-14.parquet"
)

competition_test_file = (
    DATA_DIR
    / "competition_test.parquet"
)


In [7]:
data_for_schema = pd.read_parquet(train_files[0])

excluded_columns = ["user_id", "cutoff_date", "target"]
features = [
    column
    for column in data_for_schema.columns
    if column not in excluded_columns
]

categorical_features = data_for_schema[features].select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print(f"Число признаков: {len(features)}")
print(f"Категориальные признаки: {categorical_features}")

del data_for_schema
gc.collect()

Число признаков: 629
Категориальные признаки: []


320

In [8]:
train_data = pd.concat(
    (
        pd.read_parquet(
            file,
            columns=features + ["target"],
        )
        for file in train_files
    ),
    ignore_index=True,
    copy=False,
)

valid_data = pd.read_parquet(
    validation_file,
    columns=features + ["target"],
)

print(f"Train: {train_data.shape}")
print(f"Validation: {valid_data.shape}")

Train: (1000000, 630)
Validation: (250000, 630)


In [8]:
train_data[features] = train_data[features].astype("float32")
valid_data[features] = valid_data[features].astype("float32")

y_train = train_data["target"].to_numpy(dtype="float32")
y_valid = valid_data["target"].to_numpy(dtype="float32")

y_train_log = np.log1p(y_train)
y_valid_log = np.log1p(y_valid)

assert np.all(y_train >= 0)
assert np.all(y_valid >= 0)

In [9]:
catboost_model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5,
    random_strength=0.5,
    random_seed=42,
    thread_count=-1,
    allow_writing_files=False,
)

lgbm_model = LGBMRegressor(
    objective="regression",
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    max_bin=255,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
    importance_type="gain",
)

In [10]:
catboost_model.fit(
    train_data[features],
    y_train_log,
    eval_set=(valid_data[features], y_valid_log),
    early_stopping_rounds=150,
    use_best_model=True,
    verbose=100,
)

0:	learn: 2.3175287	test: 2.3296337	best: 2.3296337 (0)	total: 866ms	remaining: 43m 17s
100:	learn: 1.7203938	test: 1.7475759	best: 1.7475759 (100)	total: 49.2s	remaining: 23m 32s
200:	learn: 1.7115792	test: 1.7433328	best: 1.7433328 (200)	total: 1m 24s	remaining: 19m 33s
300:	learn: 1.7072123	test: 1.7423379	best: 1.7423379 (300)	total: 2m 22s	remaining: 21m 18s
400:	learn: 1.7036946	test: 1.7418712	best: 1.7418712 (400)	total: 3m 28s	remaining: 22m 28s
500:	learn: 1.7003484	test: 1.7415555	best: 1.7415507 (499)	total: 3m 58s	remaining: 19m 51s
600:	learn: 1.6971124	test: 1.7413982	best: 1.7413982 (600)	total: 4m 52s	remaining: 19m 29s
700:	learn: 1.6941013	test: 1.7412937	best: 1.7412937 (700)	total: 5m 57s	remaining: 19m 31s
800:	learn: 1.6911726	test: 1.7411916	best: 1.7411914 (789)	total: 7m 1s	remaining: 19m 17s
900:	learn: 1.6882770	test: 1.7411535	best: 1.7411430 (882)	total: 8m	remaining: 18m 38s
1000:	learn: 1.6855111	test: 1.7411191	best: 1.7411121 (997)	total: 9m 6s	remaini

CatBoostRegressor(allow_writing_files=False, depth=8, eval_metric='RMSE', iterations=3000, l2_leaf_reg=5, learning_rate=0.03, loss_function='RMSE', random_seed=42, random_strength=0.5)

In [14]:
catboost_prediction_log = catboost_model.predict(
    valid_data[features]
)
catboost_prediction_log = np.maximum(catboost_prediction_log, 0)
catboost_prediction = np.expm1(catboost_prediction_log)

catboost_rmsle = root_mean_squared_log_error(
    y_valid,
    catboost_prediction,
)
pd.DataFrame(
    catboost_prediction_log
).to_csv(
    "../ensemble/validation_predictions/catboost_prediction_log.csv",
    index=False,
)

pd.DataFrame(
    catboost_prediction
).to_csv(
    "../ensemble/validation_predictions/catboost_prediction.csv",
    index=False,
)

print(f"CatBoost RMSLE: {catboost_rmsle:.6f}")
print(f"Лучшая итерация CatBoost: {catboost_model.get_best_iteration()}")

CatBoost RMSLE: 1.741073
Лучшая итерация CatBoost: 1136


In [12]:
lgbm_model.fit(
    train_data[features],
    y_train_log,
    eval_set=[(valid_data[features], y_valid_log)],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(period=100),
    ],
)

Training until validation scores don't improve for 100 rounds
[100]	valid_0's rmse: 1.7449	valid_0's l2: 3.04466
[200]	valid_0's rmse: 1.74221	valid_0's l2: 3.03529
[300]	valid_0's rmse: 1.74196	valid_0's l2: 3.03443
[400]	valid_0's rmse: 1.74202	valid_0's l2: 3.03462
Early stopping, best iteration is:
[303]	valid_0's rmse: 1.74196	valid_0's l2: 3.03441


,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.03
,n_estimators,3000
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,100


In [15]:
lgbm_prediction_log = lgbm_model.predict(
    valid_data[features],
    num_iteration=lgbm_model.best_iteration_,
)
lgbm_prediction_log = np.maximum(lgbm_prediction_log, 0)
lgbm_prediction = np.expm1(lgbm_prediction_log)

lgbm_rmsle = root_mean_squared_log_error(
    y_valid,
    lgbm_prediction,
)
pd.DataFrame(
    lgbm_prediction_log
).to_csv(
    "../ensemble/validation_predictions/lgbm_prediction_log.csv",
    index=False,
)

pd.DataFrame(
    lgbm_prediction
).to_csv(
    "../ensemble/validation_predictions/lgbm_prediction.csv",
    index=False,
)
print(f"LightGBM RMSLE: {lgbm_rmsle:.6f}")
print(f"Лучшая итерация LightGBM: {lgbm_model.best_iteration_}")

LightGBM RMSLE: 1.741956
Лучшая итерация LightGBM: 303


In [36]:
feat = pd.DataFrame({
    'features': features,
    'score': catboost_model.feature_importances_
})
feat.sort_values(ascending=False,by='score')
feat.to_csv('hren.csv', index=False)
importance = pd.read_csv("../hren.csv").sort_values(
    "score",
    ascending=False,
)

In [37]:
results = []

for top_k in [45, 100, 200, 300, 400, 500, 600]:
    selected_features = (
        importance["features"]
        .head(top_k)
        .tolist()
    )

    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        iterations=3000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=5,
        random_strength=0.5,
        random_seed=42,
        thread_count=-1,
        allow_writing_files=False,
    )

    model.fit(
        train_data[selected_features],
        y_train_log,
        eval_set=(
            valid_data[selected_features],
            y_valid_log,
        ),
        early_stopping_rounds=150,
        use_best_model=True,
        verbose=100,
    )

    prediction_log = np.maximum(
        model.predict(valid_data[selected_features]),
        0,
    )
    prediction = np.expm1(prediction_log)

    score = root_mean_squared_log_error(
        y_valid,
        prediction,
    )

    results.append({
        "top_k": top_k,
        "RMSLE": score,
        "best_iteration": model.get_best_iteration(),
    })

results = pd.DataFrame(results).sort_values("RMSLE")
results

0:	learn: 2.3175617	test: 2.3294754	best: 2.3294754 (0)	total: 187ms	remaining: 9m 22s
100:	learn: 1.7208021	test: 1.7477538	best: 1.7477538 (100)	total: 14.4s	remaining: 6m 52s
200:	learn: 1.7129960	test: 1.7437776	best: 1.7437776 (200)	total: 25.6s	remaining: 5m 56s
300:	learn: 1.7095651	test: 1.7429020	best: 1.7429020 (300)	total: 36.3s	remaining: 5m 25s
400:	learn: 1.7066013	test: 1.7425418	best: 1.7425418 (400)	total: 47.5s	remaining: 5m 7s
500:	learn: 1.7038946	test: 1.7423130	best: 1.7423130 (500)	total: 58s	remaining: 4m 49s
600:	learn: 1.7012213	test: 1.7422364	best: 1.7422364 (600)	total: 1m 9s	remaining: 4m 36s
700:	learn: 1.6985636	test: 1.7421983	best: 1.7421941 (699)	total: 1m 20s	remaining: 4m 25s
800:	learn: 1.6959234	test: 1.7422311	best: 1.7421930 (709)	total: 1m 33s	remaining: 4m 16s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 1.742193023
bestIteration = 709

Shrink model to first 710 iterations.
0:	learn: 2.3174910	test: 2.3295083	best: 2.3295

,top_k,RMSLE,best_iteration
5,500,1.741080,1078
2,200,1.741104,860
3,300,1.741229,854
6,600,1.741326,830
4,400,1.741376,927
1,100,1.741388,596
0,45,1.742193,709


In [9]:
best_top_k = 500
best_iterations = 1079

importance = (
    pd.read_csv("../hren.csv")
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)
selected_features = importance["features"].head(best_top_k).tolist()

print(f"Берём {best_top_k} признаков и {best_iterations} итераций")

Берём 500 признаков и 1079 итераций


In [10]:
for name in [
    "train_data", "valid_data", "y_train", "y_valid",
    "y_train_log", "y_valid_log", "model",
    "catboost_model", "lgbm_model",
]:
    globals().pop(name, None)
gc.collect()

# train_1 + train_2 + train_3 обучение для независимого теста.
train_for_test_files = train_files + [validation_file]
test_train = pd.concat(
    (
        pd.read_parquet(
            file,
            columns=selected_features + ["target"],
        ).astype("float32")
        for file in train_for_test_files
    ),
    ignore_index=True,
    copy=False,
)

test_data = pd.read_parquet(
    independent_test_file,
    columns=selected_features + ["target"],
).astype("float32")

# y_test_train_log = np.log1p(test_train["target"])
# y_test = test_data["target"].to_numpy()
# print(f"Train для проверки: {test_train.shape}")
# print(f"test_for_us: {test_data.shape}")

In [10]:
# test_model = CatBoostRegressor(
#     loss_function="RMSE",
#     eval_metric="RMSE",
#     iterations=best_iterations,
#     learning_rate=0.03,
#     depth=8,
#     l2_leaf_reg=5,
#     random_strength=0.5,
#     random_seed=42,
#     thread_count=-1,
#     allow_writing_files=False,
# )
# #
# test_model.fit(
#     test_train[selected_features],
#     y_test_train_log,
#     verbose=100,
# )

0:	learn: 2.3215876	total: 777ms	remaining: 13m 57s
100:	learn: 1.7292977	total: 55.8s	remaining: 9m
200:	learn: 1.7215091	total: 1m 42s	remaining: 7m 25s
300:	learn: 1.7182308	total: 2m 27s	remaining: 6m 20s
400:	learn: 1.7154825	total: 3m 12s	remaining: 5m 25s
500:	learn: 1.7128720	total: 3m 55s	remaining: 4m 31s
600:	learn: 1.7102750	total: 4m 38s	remaining: 3m 41s
700:	learn: 1.7078297	total: 5m 21s	remaining: 2m 53s
800:	learn: 1.7054304	total: 6m 4s	remaining: 2m 6s
900:	learn: 1.7032584	total: 6m 45s	remaining: 1m 20s
1000:	learn: 1.7011472	total: 8m 2s	remaining: 37.6s
1078:	learn: 1.6996060	total: 8m 52s	remaining: 0us


CatBoostRegressor(allow_writing_files=False, depth=8, eval_metric='RMSE', iterations=1079, l2_leaf_reg=5, learning_rate=0.03, loss_function='RMSE', random_seed=42, random_strength=0.5)

In [11]:
# test_prediction_log = np.maximum(
#     test_model.predict(test_data[selected_features]),
#     0,
# )
# test_prediction = np.expm1(test_prediction_log)
#
# test_rmsle = root_mean_squared_log_error(
#     y_test,
#     test_prediction,
# )
# print(f"RMSLE на test_for_us: {test_rmsle:.6f}")

RMSLE на test_for_us: 1.686630


In [11]:

final_train = pd.concat(
    [test_train, test_data],
    ignore_index=True,
    copy=False,
)

del test_train, test_data
gc.collect()

y_final_log = np.log1p(final_train["target"])

final_model_cat = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=best_iterations,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5,
    random_strength=0.5,
    random_seed=42,
    thread_count=-1,
    allow_writing_files=False,
)
final_model_lgbd = LGBMRegressor(
    objective="regression",
    n_estimators=303,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    max_bin=255,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
    importance_type="gain",
)

final_model_cat.fit(
    final_train[selected_features],
    y_final_log,
    verbose=100,
)
# final_model_lgbd.fit(
#     final_train[selected_features],
#     y_final_log,
# )

0:	learn: 2.3108268	total: 2.24s	remaining: 40m 16s
100:	learn: 1.7154415	total: 2m 27s	remaining: 23m 44s
200:	learn: 1.7082645	total: 3m 33s	remaining: 15m 32s
300:	learn: 1.7058061	total: 4m 39s	remaining: 12m 3s
400:	learn: 1.7039562	total: 5m 46s	remaining: 9m 46s
500:	learn: 1.7022969	total: 6m 53s	remaining: 7m 57s
600:	learn: 1.7006546	total: 7m 58s	remaining: 6m 20s
700:	learn: 1.6990965	total: 9m 2s	remaining: 4m 52s
800:	learn: 1.6976275	total: 10m 7s	remaining: 3m 30s
900:	learn: 1.6962343	total: 11m 10s	remaining: 2m 12s
1000:	learn: 1.6948585	total: 12m 13s	remaining: 57.2s
1078:	learn: 1.6938185	total: 13m 3s	remaining: 0us


CatBoostRegressor(allow_writing_files=False, depth=8, eval_metric='RMSE', iterations=1079, l2_leaf_reg=5, learning_rate=0.03, loss_function='RMSE', random_seed=42, random_strength=0.5)

In [12]:


competition_data = pd.read_parquet(
    competition_test_file,
    columns=["user_id"] + selected_features,
)
competition_data[selected_features] = (
    competition_data[selected_features].astype("float32")
)

competition_prediction_log_cat = np.maximum(
    final_model_cat.predict(competition_data[selected_features]),
    0,
)
# competition_prediction_log_lgbm = np.maximum(
#     final_model_lgbd.predict(competition_data[selected_features]),
#     0,
# )
competition_prediction = np.expm1(competition_prediction_log_cat)

submission = pd.DataFrame({
    "user_id": competition_data["user_id"],
    "predict": competition_prediction,
})
submission.to_csv("../ensemble/submissions/submission_cat_with_more_date.csv", index=False)
